In [1]:
# ============================================
# 🎓 GESTOR DE ESTUDIANTES - INTERFAZ PREMIUM
# Ejecuta esto en Google Colab
# ============================================

# Instalar dependencias
!pip install gradio pandas plotly -q

import gradio as gr
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from typing import List, Dict, Optional
import random

# ============================================
# FUNCIONES DE ORDENAMIENTO
# ============================================

def bubble_sort(lista: List[Dict], key: str) -> List[Dict]:
    """Bubble Sort - O(n²)"""
    arr = lista.copy()
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j][key] > arr[j + 1][key]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

def insertion_sort(lista: List[Dict], key: str) -> List[Dict]:
    """Insertion Sort - O(n²)"""
    arr = lista.copy()
    for i in range(1, len(arr)):
        actual = arr[i]
        j = i - 1
        while j >= 0 and arr[j][key] > actual[key]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = actual
    return arr

def quicksort(lista: List[Dict], key: str) -> List[Dict]:
    """Quick Sort - O(n log n)"""
    if len(lista) <= 1:
        return lista
    pivot = lista[len(lista) // 2]
    pivot_val = pivot[key]
    menores = [x for x in lista if x[key] < pivot_val]
    iguales = [x for x in lista if x[key] == pivot_val]
    mayores = [x for x in lista if x[key] > pivot_val]
    return quicksort(menores, key) + iguales + quicksort(mayores, key)

# ============================================
# FUNCIONES DE BÚSQUEDA
# ============================================

def busqueda_lineal(lista: List[Dict], valor, key: str) -> Optional[Dict]:
    """Búsqueda Lineal - O(n)"""
    valor = valor.lower() if isinstance(valor, str) else valor
    for item in lista:
        dato = item[key].lower() if isinstance(item[key], str) else item[key]
        if dato == valor:
            return item
    return None

def busqueda_binaria(lista: List[Dict], valor, key: str) -> Optional[Dict]:
    """Búsqueda Binaria - O(log n)"""
    lista_ordenada = quicksort(lista, key)
    izquierda, derecha = 0, len(lista_ordenada) - 1
    valor = valor.lower() if isinstance(valor, str) else valor

    while izquierda <= derecha:
        medio = (izquierda + derecha) // 2
        actual = lista_ordenada[medio]
        dato = actual[key].lower() if isinstance(actual[key], str) else actual[key]

        if dato == valor:
            return actual
        elif dato < valor:
            izquierda = medio + 1
        else:
            derecha = medio - 1
    return None

# ============================================
# GESTIÓN DE DATOS
# ============================================

class GestorEstudiantes:
    def __init__(self):
        self.estudiantes = [
            {"id": 1, "nombre": "Ana García", "edad": 20, "promedio": 4.5, "semestre": 3},
            {"id": 2, "nombre": "Carlos López", "edad": 22, "promedio": 4.8, "semestre": 5},
            {"id": 3, "nombre": "Luis Martínez", "edad": 19, "promedio": 3.9, "semestre": 2},
            {"id": 4, "nombre": "María Rodríguez", "edad": 21, "promedio": 4.2, "semestre": 4},
            {"id": 5, "nombre": "Pedro Sánchez", "edad": 23, "promedio": 3.7, "semestre": 6},
        ]

    def obtener_df(self):
        """Retorna DataFrame estilizado"""
        df = pd.DataFrame(self.estudiantes)
        df.columns = ['ID', 'Nombre', 'Edad', 'Promedio', 'Semestre']
        return df

    def crear_grafico_promedios(self):
        """Crea gráfico de barras de promedios"""
        df = pd.DataFrame(self.estudiantes)

        # Colores según promedio
        colores = ['#10b981' if p >= 4.5 else '#f59e0b' if p >= 3.5 else '#ef4444'
                   for p in df['promedio']]

        fig = go.Figure(data=[
            go.Bar(
                x=df['nombre'],
                y=df['promedio'],
                marker_color=colores,
                text=df['promedio'].round(2),
                textposition='outside',
                hovertemplate='<b>%{x}</b><br>Promedio: %{y:.2f}<extra></extra>'
            )
        ])

        fig.update_layout(
            title={
                'text': '📊 Promedios de Estudiantes',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 20, 'color': '#667eea', 'family': 'Arial Black'}
            },
            xaxis_title='Estudiante',
            yaxis_title='Promedio',
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            font=dict(size=12),
            height=400,
            yaxis=dict(range=[0, 5.5], gridcolor='#e5e7eb'),
            xaxis=dict(gridcolor='#e5e7eb'),
            showlegend=False
        )

        return fig

    def crear_grafico_distribucion(self):
        """Crea gráfico de distribución por semestre"""
        df = pd.DataFrame(self.estudiantes)
        semestre_counts = df['semestre'].value_counts().sort_index()

        fig = go.Figure(data=[
            go.Scatter(
                x=semestre_counts.index,
                y=semestre_counts.values,
                mode='lines+markers',
                line=dict(color='#667eea', width=3),
                marker=dict(size=12, color='#764ba2'),
                fill='tozeroy',
                fillcolor='rgba(102, 126, 234, 0.2)',
                hovertemplate='<b>Semestre %{x}</b><br>Estudiantes: %{y}<extra></extra>'
            )
        ])

        fig.update_layout(
            title={
                'text': '📈 Distribución por Semestre',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 20, 'color': '#667eea', 'family': 'Arial Black'}
            },
            xaxis_title='Semestre',
            yaxis_title='Cantidad de Estudiantes',
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            height=400,
            xaxis=dict(gridcolor='#e5e7eb'),
            yaxis=dict(gridcolor='#e5e7eb'),
            showlegend=False
        )

        return fig

    def agregar_estudiante(self, nombre: str, edad: int, promedio: float, semestre: int):
        """Agrega un nuevo estudiante"""
        if not nombre or nombre.strip() == "":
            return "❌ El nombre no puede estar vacío", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

        try:
            edad = int(edad)
            promedio = float(promedio)
            semestre = int(semestre)

            if promedio < 0 or promedio > 5:
                return "❌ El promedio debe estar entre 0 y 5", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

            if edad < 15 or edad > 100:
                return "❌ La edad debe estar entre 15 y 100", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

            nuevo_id = max([e['id'] for e in self.estudiantes], default=0) + 1
            nuevo = {
                "id": nuevo_id,
                "nombre": nombre.strip(),
                "edad": edad,
                "promedio": promedio,
                "semestre": semestre
            }

            self.estudiantes.append(nuevo)
            return f"✅ ¡Estudiante '{nombre}' agregado exitosamente con ID #{nuevo_id}!", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

        except ValueError as e:
            return f"❌ Error: Verifica que los datos sean válidos ({str(e)})", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

    def ordenar(self, algoritmo: str, campo: str):
        """Ordena estudiantes"""
        campo_map = {
            "Nombre": "nombre",
            "Edad": "edad",
            "Promedio": "promedio",
            "Semestre": "semestre",
            "ID": "id"
        }

        campo_real = campo_map.get(campo, "nombre")

        if algoritmo == "🔵 Bubble Sort":
            self.estudiantes = bubble_sort(self.estudiantes, campo_real)
            msg = "🔵 Ordenado con Bubble Sort O(n²)"
        elif algoritmo == "📥 Insertion Sort":
            self.estudiantes = insertion_sort(self.estudiantes, campo_real)
            msg = "📥 Ordenado con Insertion Sort O(n²)"
        else:
            self.estudiantes = quicksort(self.estudiantes, campo_real)
            msg = "⚡ Ordenado con Quick Sort O(n log n)"

        return f"{msg} por {campo}", self.obtener_df(), self.crear_grafico_promedios()

    def buscar(self, tipo: str, campo: str, valor: str):
        """Busca un estudiante"""
        if not valor or valor.strip() == "":
            return "❌ Ingresa un valor para buscar"

        campo_map = {"Nombre": "nombre", "ID": "id"}
        campo_real = campo_map.get(campo, "nombre")

        if campo == "ID":
            try:
                valor = int(valor)
            except ValueError:
                return "❌ El ID debe ser un número"

        if tipo == "🔄 Búsqueda Lineal":
            resultado = busqueda_lineal(self.estudiantes, valor, campo_real)
            tipo_msg = "🔄 Búsqueda Lineal O(n)"
        else:
            resultado = busqueda_binaria(self.estudiantes, valor, campo_real)
            tipo_msg = "⚡ Búsqueda Binaria O(log n)"

        if resultado:
            badge = "🟢" if resultado['promedio'] >= 4.5 else "🟡" if resultado['promedio'] >= 3.5 else "🔴"
            return f"""
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
    <h2 style='margin: 0 0 20px 0;'>✅ Estudiante Encontrado</h2>
    <p style='margin: 5px 0; opacity: 0.9;'><strong>{tipo_msg}</strong></p>
    <hr style='border: 1px solid rgba(255,255,255,0.3); margin: 15px 0;'>
    <p style='font-size: 18px; margin: 10px 0;'><strong>🆔 ID:</strong> #{resultado['id']}</p>
    <p style='font-size: 18px; margin: 10px 0;'><strong>👤 Nombre:</strong> {resultado['nombre']}</p>
    <p style='font-size: 18px; margin: 10px 0;'><strong>🎂 Edad:</strong> {resultado['edad']} años</p>
    <p style='font-size: 18px; margin: 10px 0;'><strong>📊 Promedio:</strong> {badge} {resultado['promedio']}</p>
    <p style='font-size: 18px; margin: 10px 0;'><strong>📚 Semestre:</strong> {resultado['semestre']}</p>
</div>
"""
        else:
            return f"""
<div style='background: #fee2e2; padding: 20px; border-radius: 10px; border-left: 5px solid #ef4444;'>
    <h3 style='color: #991b1b; margin: 0;'>❌ No se encontró el estudiante</h3>
    <p style='color: #7f1d1d; margin: 10px 0 0 0;'>No existe ningún estudiante con {campo}: <strong>{valor}</strong></p>
</div>
"""

    def eliminar_estudiante(self, id_estudiante: int):
        """Elimina un estudiante"""
        try:
            id_num = int(id_estudiante)
            estudiante = next((e for e in self.estudiantes if e['id'] == id_num), None)

            if estudiante:
                self.estudiantes = [e for e in self.estudiantes if e['id'] != id_num]
                return f"✅ Estudiante '{estudiante['nombre']}' (ID #{id_num}) eliminado correctamente", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()
            else:
                return f"❌ No existe estudiante con ID #{id_num}", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()
        except ValueError:
            return "❌ El ID debe ser un número válido", self.obtener_df(), self.crear_grafico_promedios(), self.crear_grafico_distribucion()

    def obtener_estadisticas(self):
        """Estadísticas con diseño mejorado"""
        if not self.estudiantes:
            return "<div style='padding: 40px; text-align: center; color: #9ca3af;'><h2>📚 No hay estudiantes registrados</h2></div>"

        total = len(self.estudiantes)
        promedio_general = sum(e['promedio'] for e in self.estudiantes) / total
        mejor = max(self.estudiantes, key=lambda x: x['promedio'])
        peor = min(self.estudiantes, key=lambda x: x['promedio'])
        aprobados = sum(1 for e in self.estudiantes if e['promedio'] >= 3.0)

        return f"""
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 20px; color: white; box-shadow: 0 15px 40px rgba(0,0,0,0.3);'>
    <h1 style='margin: 0 0 25px 0; font-size: 32px;'>📊 Estadísticas Generales</h1>

    <div style='display: grid; grid-template-columns: repeat(2, 1fr); gap: 20px; margin-bottom: 20px;'>
        <div style='background: rgba(255,255,255,0.2); padding: 20px; border-radius: 15px; backdrop-filter: blur(10px);'>
            <div style='font-size: 40px; font-weight: bold;'>{total}</div>
            <div style='opacity: 0.9; font-size: 16px;'>👥 Total Estudiantes</div>
        </div>
        <div style='background: rgba(255,255,255,0.2); padding: 20px; border-radius: 15px; backdrop-filter: blur(10px);'>
            <div style='font-size: 40px; font-weight: bold;'>{promedio_general:.2f}</div>
            <div style='opacity: 0.9; font-size: 16px;'>📈 Promedio General</div>
        </div>
        <div style='background: rgba(255,255,255,0.2); padding: 20px; border-radius: 15px; backdrop-filter: blur(10px);'>
            <div style='font-size: 40px; font-weight: bold;'>{aprobados}</div>
            <div style='opacity: 0.9; font-size: 16px;'>✅ Aprobados (≥3.0)</div>
        </div>
        <div style='background: rgba(255,255,255,0.2); padding: 20px; border-radius: 15px; backdrop-filter: blur(10px);'>
            <div style='font-size: 40px; font-weight: bold;'>{total - aprobados}</div>
            <div style='opacity: 0.9; font-size: 16px;'>❌ Reprobados</div>
        </div>
    </div>

    <div style='background: rgba(255,255,255,0.2); padding: 20px; border-radius: 15px; backdrop-filter: blur(10px); margin-top: 20px;'>
        <h3 style='margin: 0 0 15px 0;'>🏆 Destacados</h3>
        <p style='margin: 8px 0; font-size: 16px;'><strong>🥇 Mejor estudiante:</strong> {mejor['nombre']} ({mejor['promedio']})</p>
        <p style='margin: 8px 0; font-size: 16px;'><strong>📉 Promedio más bajo:</strong> {peor['nombre']} ({peor['promedio']})</p>
    </div>
</div>
"""

# Crear instancia
gestor = GestorEstudiantes()

# ============================================
# TEMA PERSONALIZADO
# ============================================

tema_custom = gr.themes.Soft(
    primary_hue="purple",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "Arial", "sans-serif"],
).set(
    button_primary_background_fill="linear-gradient(135deg, #667eea 0%, #764ba2 100%)",
    button_primary_background_fill_hover="linear-gradient(135deg, #764ba2 0%, #667eea 100%)",
    block_title_text_weight="700",
    block_border_width="2px",
    block_shadow="0 10px 30px rgba(0,0,0,0.1)",
)

# ============================================
# INTERFAZ GRADIO PREMIUM
# ============================================

css = """
.gradio-container {
    font-family: 'Inter', sans-serif !important;
}
.gr-button {
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: all 0.3s !important;
}
.gr-button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 10px 25px rgba(102, 126, 234, 0.3) !important;
}
.gr-form {
    border-radius: 15px !important;
    box-shadow: 0 5px 20px rgba(0,0,0,0.1) !important;
}
h1, h2, h3 {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
}
"""

with gr.Blocks(theme=tema_custom, css=css, title="🎓 Gestor de Estudiantes Premium") as app:

    gr.Markdown("""
    <div style='text-align: center; padding: 20px;'>
        <h1 style='font-size: 48px; margin-bottom: 10px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>
            🎓 Gestor de Estudiantes
        </h1>
        <p style='font-size: 20px; color: #64748b; margin: 0;'>
            Sistema Inteligente con Algoritmos de Ordenamiento y Búsqueda
        </p>
    </div>
    """)

    with gr.Tabs():
        # ============================================
        # TAB 1: GESTIÓN
        # ============================================
        with gr.Tab("📝 Gestión de Estudiantes"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### ➕ Agregar Nuevo Estudiante")
                    nombre_input = gr.Textbox(
                        label="📝 Nombre Completo",
                        placeholder="Ej: Juan Pérez González",
                        lines=1
                    )
                    with gr.Row():
                        edad_input = gr.Number(
                            label="🎂 Edad",
                            value=20,
                            minimum=15,
                            maximum=100
                        )
                        semestre_input = gr.Number(
                            label="📚 Semestre",
                            value=1,
                            minimum=1,
                            maximum=12
                        )
                    promedio_input = gr.Slider(
                        label="📊 Promedio",
                        minimum=0,
                        maximum=5,
                        step=0.1,
                        value=4.0
                    )
                    btn_agregar = gr.Button(
                        "✅ Agregar Estudiante",
                        variant="primary",
                        size="lg"
                    )
                    mensaje_agregar = gr.Markdown()

                with gr.Column(scale=2):
                    gr.Markdown("### 📋 Lista de Estudiantes Registrados")
                    tabla = gr.DataFrame(
                        value=gestor.obtener_df(),
                        label="",
                        interactive=False,
                        wrap=True
                    )

            gr.Markdown("---")

            with gr.Row():
                grafico_promedios = gr.Plot(label="📊 Visualización de Promedios")
                grafico_distribucion = gr.Plot(label="📈 Distribución por Semestre")

        # ============================================
        # TAB 2: ORDENAMIENTO
        # ============================================
        with gr.Tab("🔄 Ordenamiento"):
            gr.Markdown("""
            ### 🎯 Algoritmos de Ordenamiento
            Selecciona el campo y el algoritmo para ordenar la lista de estudiantes
            """)

            with gr.Row():
                with gr.Column():
                    campo_select = gr.Radio(
                        choices=["Nombre", "Edad", "Promedio", "Semestre", "ID"],
                        value="Nombre",
                        label="📌 Ordenar por",
                        info="Selecciona el campo por el cual deseas ordenar"
                    )

                with gr.Column():
                    algoritmo_select = gr.Radio(
                        choices=["🔵 Bubble Sort", "📥 Insertion Sort", "⚡ Quick Sort"],
                        value="⚡ Quick Sort",
                        label="⚙️ Algoritmo",
                        info="Cada algoritmo tiene diferente complejidad"
                    )

            btn_ordenar = gr.Button("🔄 Ordenar Ahora", variant="primary", size="lg")
            mensaje_ordenar = gr.Markdown()
            tabla_ordenada = gr.DataFrame(value=gestor.obtener_df(), interactive=False)
            grafico_ordenado = gr.Plot()

        # ============================================
        # TAB 3: BÚSQUEDA
        # ============================================
        with gr.Tab("🔍 Búsqueda"):
            gr.Markdown("""
            ### 🎯 Algoritmos de Búsqueda
            Encuentra estudiantes usando búsqueda lineal o binaria
            """)

            with gr.Row():
                with gr.Column():
                    tipo_busqueda = gr.Radio(
                        choices=["🔄 Búsqueda Lineal", "⚡ Búsqueda Binaria"],
                        value="🔄 Búsqueda Lineal",
                        label="🔎 Tipo de Búsqueda",
                        info="Lineal: O(n) | Binaria: O(log n)"
                    )

                with gr.Column():
                    campo_busqueda = gr.Radio(
                        choices=["Nombre", "ID"],
                        value="Nombre",
                        label="📌 Buscar por"
                    )

            valor_busqueda = gr.Textbox(
                label="🔍 Valor a buscar",
                placeholder="Ingresa el nombre o ID del estudiante...",
                lines=1
            )
            btn_buscar = gr.Button("🔍 Buscar Ahora", variant="primary", size="lg")
            resultado_busqueda = gr.HTML()

        # ============================================
        # TAB 4: ESTADÍSTICAS
        # ============================================
        with gr.Tab("📊 Estadísticas"):
            gr.Markdown("### 📈 Panel de Control y Estadísticas")
            btn_stats = gr.Button("🔄 Actualizar Estadísticas", variant="primary", size="lg")
            stats_output = gr.HTML(value=gestor.obtener_estadisticas())

            with gr.Row():
                stats_grafico1 = gr.Plot(value=gestor.crear_grafico_promedios())
                stats_grafico2 = gr.Plot(value=gestor.crear_grafico_distribucion())

        # ============================================
        # TAB 5: ELIMINAR
        # ============================================
        with gr.Tab("🗑️ Eliminar"):
            gr.Markdown("""
            ### ⚠️ Eliminar Estudiante
            Ingresa el ID del estudiante que deseas eliminar
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    id_eliminar = gr.Number(
                        label="🆔 ID del Estudiante",
                        value=1,
                        minimum=1
                    )
                    btn_eliminar = gr.Button(
                        "🗑️ Eliminar Estudiante",
                        variant="stop",
                        size="lg"
                    )
                    mensaje_eliminar = gr.Markdown()

                with gr.Column(scale=2):
                    tabla_eliminar = gr.DataFrame(value=gestor.obtener_df(), interactive=False)

    # ============================================
    # EVENTOS
    # ============================================

    btn_agregar.click(
        fn=gestor.agregar_estudiante,
        inputs=[nombre_input, edad_input, promedio_input, semestre_input],
        outputs=[mensaje_agregar, tabla, grafico_promedios, grafico_distribucion]
    ).then(
        fn=lambda: ("", 20, 4.0, 1),
        outputs=[nombre_input, edad_input, promedio_input, semestre_input]
    )

    btn_ordenar.click(
        fn=gestor.ordenar,
        inputs=[algoritmo_select, campo_select],
        outputs=[mensaje_ordenar, tabla_ordenada, grafico_ordenado]
    )

    btn_buscar.click(
        fn=gestor.buscar,
        inputs=[tipo_busqueda, campo_busqueda, valor_busqueda],
        outputs=[resultado_busqueda]
    )

    btn_eliminar.click(
        fn=gestor.eliminar_estudiante,
        inputs=[id_eliminar],
        outputs=[mensaje_eliminar, tabla_eliminar, stats_grafico1, stats_grafico2]
    )

    btn_stats.click(
        fn=lambda: [gestor.obtener_estadisticas(), gestor.crear_grafico_promedios(), gestor.crear_grafico_distribucion()],
        outputs=[stats_output, stats_grafico1, stats_grafico2]
    )

# ============================================
# EJECUTAR
# ============================================
print("🚀 Iniciando Gestor de Estudiantes Premium...")
print("✨ Interfaz moderna y profesional")
print("📊 Con gráficos interactivos")
app.launch(share=True, debug=True)

🚀 Iniciando Gestor de Estudiantes Premium...
✨ Interfaz moderna y profesional
📊 Con gráficos interactivos
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a417587a851cbaeab2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a417587a851cbaeab2.gradio.live
